# Chapter 5 &mdash; Verify by Constructing Twice

**Concept 4 of the Chapter 5 decomposition:** *Verification by Construction Twice: Minimal DFA Uniqueness and `iso_dfa`*

Design from two perspectives, minimize both, and check isomorphism &mdash; Myhill&ndash;Nerode guarantees they must match.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Verification-By-Construction-Twice/Concept-Verification-By-Construction-Twice.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The strongest cheap check available: **build the DFA twice from two different
perspectives**, minimize both, and ask whether they are **isomorphic**.

Myhill&ndash;Nerode (Chapter 6) says the minimal DFA for a regular language is **unique up
to renaming**. So:

* two correct designs $\Rightarrow$ `iso_dfa(min_dfa(A), min_dfa(B))` is `True`;
* `False` $\Rightarrow$ **at least one design is wrong**, and you must find out which.

This turns "I think it's right" into a decidable question.

## 2. Definitions

### Perspective 1: track the parity pair directly

In [ ]:
A = md2mc('''DFA
IF : 0 -> P0
IF : 1 -> P1
P0 : 0 -> IF
P0 : 1 -> P01
P1 : 0 -> P01
P1 : 1 -> IF
P01: 0 -> P1
P01: 1 -> P0
''')

### Perspective 2: even length, then even zeros &mdash; a different decomposition

In [ ]:
B = md2mc('''DFA
IF  : 0 -> Od0     !! odd length, odd 0s
IF  : 1 -> Od1     !! odd length, even 0s
Od0 : 0 -> IF
Od0 : 1 -> Ev2
Od1 : 0 -> Ev2
Od1 : 1 -> IF
Ev2 : 0 -> Od1
Ev2 : 1 -> Od0
''')

### A deliberately WRONG third design, to see the check fire

In [ ]:
Bad = md2mc('''DFA
IF : 0 -> P0
IF : 1 -> IF     !! BUG: 1s ignored
P0 : 0 -> IF
P0 : 1 -> P0
''')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch5&nbsp;3.&nbsp;Best Practices: Mnemonic State Names and Documented Transitions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Mnemonic-State-Names/Concept-Mnemonic-State-Names.ipynb) &nbsp;&middot;&nbsp; [**Chapter 5** index](https://github.com/ganeshutah/Jove/blob/master/Chapter5-DFADsg/README.md) &nbsp;&middot;&nbsp; [Ch5&nbsp;5.&nbsp;Sliding-Window Conditions: "Every Block of 3 Has Exactly Two 1s"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Sliding-Window/Concept-Sliding-Window.ipynb)&nbsp;&rarr;

---

## 3. Tests

Two correct designs minimize to isomorphic machines.

In [ ]:
mA, mB = min_dfa(A), min_dfa(B)
print("|Q| after minimization : A=%d  B=%d" % (len(mA["Q"]), len(mB["Q"])))
print("isomorphic? ", iso_dfa(mA, mB))
assert iso_dfa(mA, mB)
print("\nMyhill-Nerode: the minimal DFA is unique, so this HAD to hold.")

The wrong design is caught &mdash; and `langeq_dfa` produces a counterexample string.

In [ ]:
print("Bad isomorphic to A? ", iso_dfa(min_dfa(Bad), mA))
print("Bad language-equal?  ", langeq_dfa(Bad, A))
assert not langeq_dfa(Bad, A)
from itertools import product
wit = next(''.join(p) for k in range(6) for p in product('01', repeat=k)
           if accepts_dfa(Bad, ''.join(p)) != accepts_dfa(A, ''.join(p)))
print("shortest witness :", repr(wit),
      " A says", accepts_dfa(A, wit), " Bad says", accepts_dfa(Bad, wit))

Note the two checks answer different questions.

In [ ]:
print("langeq_dfa : same LANGUAGE (works on any two DFA)")
print("iso_dfa    : same SHAPE up to renaming (meaningful after min_dfa)")
print()
print("A vs B before minimizing: langeq=%s iso=%s"
      % (langeq_dfa(A, B), iso_dfa(A, B)))

## 4. Exercises


1. Design a third perspective on the same language and check all three.
2. Why is `iso_dfa` on *non*-minimal DFA a weak test?
3. When `iso_dfa` fails, how do you tell which of the two designs is wrong?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter5-DFADsg/Concept-Verification-By-Construction-Twice')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')